# Preprocessing

This tutorial covers the preprocessing steps of TF-MInDi:
- Loading motif collections and annotations
- Extracting seqlets from contribution scores
- Calculating motif similarities
- Creating and saving tfmindi's custom {class}anndata.Anndata objects

## Data Requirements

To get started with `tfmindi`, we need three types of data:
- a motif database of known transcription factor motifs
- one hot encoded sequences for genomic regions of interest
- attribution scores representing nucleotide importances for these same regions from a trained neural network.  

`tfmindi` provides functionality to fetch SCENIC+ motif collections if you don't have your own available.  
Getting attribution scores and one-hot encoded regions is outside the scope of `tfmindi`. We recommend using CREsted for this (i.e. with [this function](https://crested.readthedocs.io/en/latest/api/tools/_autosummary/crested.tl.contribution_scores_specific.html)), yet any tool that can do this works fine (such as [tangermeme](https://tangermeme.readthedocs.io/en/latest/tutorials/Tutorial_A3_Deep_LIFT_SHAP.html)). So long as you can load these into python as numpy arrays.

## Download tutorial data

### Contribution scores

Contribution score data to reproduce this tutorial is available on Zenodo using doi: [10.5281/zenodo.18757793](https://doi.org/10.5281/zenodo.18757793). These are contribution scores generated using the [deepBICCN model](https://crested.readthedocs.io/en/latest/models/BICCN/deepbiccn2.html), see [CREsted tutorial for more info](https://crested.readthedocs.io/en/latest/tutorials/enhancer_code_analysis.html).

In [2]:
!wget https://zenodo.org/records/18757794/files/tutorial_data.tar.gz

--2026-02-24 18:21:41--  https://zenodo.org/records/18757794/files/tutorial_data.tar.gz
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.184.98.114, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1654120666 (1.5G) [application/octet-stream]
Saving to: ‘tutorial_data.tar.gz’

tutorial_data.tar.g 100%[===================>]   1.54G   103MB/s    in 17s     

2026-02-24 18:21:58 (95.2 MB/s) - ‘tutorial_data.tar.gz’ saved [1654120666/1654120666]



In [4]:
!tar -xvf tutorial_data.tar.gz

tutorial_data/
tutorial_data/modisco_results_ft_2000/
tutorial_data/modisco_results_ft_2000/Astro_oh.npz
tutorial_data/modisco_results_ft_2000/SstChodl_contrib.npz
tutorial_data/modisco_results_ft_2000/Sncg_oh.npz
tutorial_data/modisco_results_ft_2000/Micro_PVM_contrib.npz
tutorial_data/modisco_results_ft_2000/L6b_modisco_results.h5
tutorial_data/modisco_results_ft_2000/Micro_PVM_oh.npz
tutorial_data/modisco_results_ft_2000/L5_6NP_contrib.npz
tutorial_data/modisco_results_ft_2000/Lamp5_modisco_results.h5
tutorial_data/modisco_results_ft_2000/Vip_contrib.npz
tutorial_data/modisco_results_ft_2000/OPC_modisco_results.h5
tutorial_data/modisco_results_ft_2000/VLMC_oh.npz
tutorial_data/modisco_results_ft_2000/SstChodl_modisco_results.h5
tutorial_data/modisco_results_ft_2000/Sncg_modisco_results.h5
tutorial_data/modisco_results_ft_2000/Vip_oh.npz
tutorial_data/modisco_results_ft_2000/L6IT_oh.npz
tutorial_data/modisco_results_ft_2000/Oligo_modisco_results.h5
tutorial_data/modisco_results_ft_20

### Sampled motifs

To reduce the number of TF-MINDI features 
(decreasing run time and memory requirements) we clustered our motif collection
and from each cluster sampled 40 motifs.
Let's download a text file with those sampled motifs.
Note, the use of this sampled collection is optional.

In [5]:
!wget https://raw.githubusercontent.com/aertslab/TF-MINDI/refs/heads/main/paper/sampled_motifs.txt

--2026-02-24 18:26:41--  https://raw.githubusercontent.com/aertslab/TF-MINDI/refs/heads/main/paper/sampled_motifs.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 105797 (103K) [text/plain]
Saving to: ‘sampled_motifs.txt’

sampled_motifs.txt  100%[===================>] 103.32K  --.-KB/s    in 0.01s   

2026-02-24 18:26:42 (9.57 MB/s) - ‘sampled_motifs.txt’ saved [105797/105797]



## Fetching motif collections and annotations

On top of the contribution scores and sequences, we'll need motif collections to calculate similarity scores using {func}`~tfmindi.pp.calculate_motif_similarity` with the extracted seqlets.  
TF-MInDi provides functionality to fetch and load some of the default SCENIC+ motif databases.  



The TF-MINDI motif collection (v11 Aerts lab motif collection) contains ~70k motifs. This is a meta-database of motifs covering many motif collections (e.g. Jaspar, Cis-BP, Hocomoco, ...).
For a large subset of these motifs we gather TF-annotations (from experimental evidence).

Next, we clustered the motif collection based on pairwise motif similarities and annotated clusters to TF-families using the [AnimalTFDB4 resource](https://guolab.wchscu.cn/AnimalTFDB4/#/). This was done one multiple clustering resolutions (see `Annotation for cluster resolutions`).

To cluster and annotate seqlets, similarities for each seqlet to the reference motif collection will be calculated.
To make this procedure more efficient, the motif collection was downsampled by selecting x-number of motifs per cluster. (see `PCA data for number of motifs per cluster`).


In [1]:
import tfmindi as tm

import os
import re
import numpy as np
from tqdm import tqdm

In [2]:
# TODO: fetch motif collection data from webserver.
motif_collection = tm.MotifCollectionData("mcv11.refdata.tar.gz")

motif_collection


Motif Collection Data

Archive file path:
	mcv11.refdata.tar.gz
Annotation for cluster resolutions:
0.1
	0.5
	0.8
	1.0
	1.5
	2.0
	2.5
	3.0
	3.5
PCA data for number of motifs per cluster:
10
	20
	50
	5

Let's load the motifs, we will use the downsampled data containing 20 motifs per clusters.

In [3]:
motif_collection_motifs = motif_collection.get_motifs(20)

In [4]:
list(motif_collection_motifs.keys())[0:5]

[('', 'homer/homer__NWTAAYCYAATCAWN_DUX4.cb'),
 ('', 'homer/homer__CNGTCCTCCC_Znf263.cb'),
 ('', '-0x67e79051c947ed4d'),
 ('', '0x79b4018a7a2595bc'),
 ('', 'homer/homer__TAAYCYAATCAA_Duxbl.cb')]

In [5]:
motif_collection_motifs[('', 'homer/homer__NWTAAYCYAATCAWN_DUX4.cb')].shape

(4, 15)

In [6]:
len(motif_collection_motifs)

3626

## Extracting seqlets using tangermeme

We use `tangermeme` to extract seqlets (spans of nucleotides with high importance scores) from our nucleotide level contribution scores per cell type coming from a CREsted model.  
We calculated these scores for cell-type specific enhancer regions only so we will be able to link each region back to a specific cell type, but this is not required to run `tfmindi`.  

To extract seqlets, we use the {func}`~tfmindi.pp.extract_seqlets` function, which wraps `tangermeme`s [recursive_seqlets](https://tangermeme.readthedocs.io/en/latest/tutorials/Tutorial_A4_Seqlets.html#Recursive-Seqlets) functionality.  
The resulting seqlets will be scaled to a range of [-1,1] and sign corrected so the average contribution values are always positive. 

The expected input shape of the contribution scores and one hot-encoded regions is (N, 4, W).  

In [7]:
# extract_seqlets expects the inputs to be in shape (n, 4, region_width), so we concatenate the cell type specific contributions
CONTRIB_FOLDER = "tutorial_data/modisco_results_ft_2000/"

contrib_list = []
oh_list = []
classes_list = []

# getting cell type names from file names (optional, not required to run tfmindi)
class_names = [
    re.match(r"(.+?)_oh\.npz$", f).group(1)  # type: ignore
    for f in os.listdir(CONTRIB_FOLDER)
    if f.endswith("_oh.npz")
]

for i, c in enumerate(tqdm(class_names)):
    contrib_list.append(np.load(os.path.join(CONTRIB_FOLDER, f"{c}_contrib.npz"))["arr_0"])
    oh_list.append(np.load(os.path.join(CONTRIB_FOLDER, f"{c}_oh.npz"))["arr_0"])
    classes_list.append(np.repeat(c, oh_list[i].shape[0]))

oh = np.concatenate(oh_list)
contrib = np.concatenate(contrib_list)
classes = np.concatenate(classes_list)  # not required to run tfmindi, but useful for interpretation later
region_id_to_ct_map = {
    i: str(c) for i, c in enumerate(classes)
}  # not required to run tfmindi, but useful for interpretation later

100%|██████████| 19/19 [00:21<00:00,  1.11s/it]


In [8]:
# extract seqlets from the contributions and one-hot encoded sequences
seqlets_df, seqlets_matrices = tm.pp.extract_seqlets(
    contrib=contrib,
    oh=oh,
    threshold=0.05,  # importance threshold for seqlet extraction. Lower = fewer seqlets.
    additional_flanks=3,  # flanking bases to include around seqlet
)

Processing seqlets: 100%|██████████| 679653/679653 [00:13<00:00, 52113.61it/s]


In [9]:
len(seqlets_matrices)

679653

In [10]:
seqlets_df.head(3)  # information on seqlet position and significance

,example_idx,start,end,attribution,p-value
0,18770,1052,1065,22.867273,2.963802e-15
1,19895,1103,1134,42.865262,2.971223e-15
2,19813,1045,1059,13.644972,4.303625e-15


In [11]:
seqlets_matrices[0].shape  # seqlets_matrices is a list with len(seqlets), that contains the scaled matrix per seqlet

(4, 13)

## Calculating motif similarity

Next, we calculate similarity scores using {func}`~tfmindi.pp.calculate_motif_similarity` between the extracted seqlets and the TF-MINDI motif collection by using `memelite`'s TomTom implementation.  
The resulting similarity matrix will be log-transformed and negated before performing clustering.  

In [12]:
sim_matrix = tm.pp.calculate_motif_similarity(
    seqlets_matrices,
    motif_collection_motifs,
    chunk_size=50000,  # you can supply a seqlet chunk_size if you have memory constraints
    n_nearest=100,  # you can limit the number of nearest motif similarities to store per seqlet to reduce memory usage
)

Processing chunks: 100%|██████████| 14/14 [10:07<00:00, 43.42s/it]


In [13]:
print(sim_matrix.shape)  # (n_seqlets, n_motifs)

(679653, 3626)


We can store the data objects we have processed so far together in a single Anndata object.  
This anndata object will become the input for all our `tfmindi.tl` tooling and `tfmindi.pl` plotting functions.  
The motif annotations etc are not mandatory but can be used to create visualisations later.  

In [14]:
adata = tm.pp.create_seqlet_adata(
    sim_matrix,  # mandatory
    seqlets_df,  # mandatory
    seqlet_matrices=seqlets_matrices,
    oh_sequences=oh,
    contrib_scores=contrib,
    motif_collection=motif_collection_motifs
)
adata

AnnData object with n_obs × n_vars = 679653 × 3626
    obs: 'example_idx', 'start', 'end', 'attribution', 'p-value', 'seqlet_matrix', 'seqlet_oh', 'example_oh_idx', 'example_contrib_idx'
    var: 'motif_ppm'
    uns: 'unique_examples'

In [15]:
# optional: let's also add a "cell_type" column to the adata.obs based on our classes and example_idx (the region's idx)
adata.obs["cell_type"] = adata.obs["example_idx"].map(region_id_to_ct_map).astype("category")

In [16]:
adata.obs.head(3)

,example_idx,start,end,attribution,p-value,seqlet_matrix,seqlet_oh,example_oh_idx,example_contrib_idx,cell_type
0,18770,1052,1065,22.867273,2.963802e-15,"[[-0.38142416, -0.21325767, 0.56363165, -0.447...","[[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,...",0,0,Oligo
1,19895,1103,1134,42.865262,2.971223e-15,"[[0.9126998, 0.572017, 0.33544934, 0.30004725,...","[[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",1,1,Oligo
2,19813,1045,1059,13.644972,4.303625e-15,"[[0.18059267, -0.007956118, -0.34745765, -0.09...","[[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0,...",2,2,Oligo


In [17]:
adata.var.head(3)

,motif_ppm
homer/homer__NWTAAYCYAATCAWN_DUX4.cb,"[[0.217, 0.3183183, 0.233, 0.997, 0.997, 0.201..."
homer/homer__CNGTCCTCCC_Znf263.cb,"[[0.152, 0.333, 0.005, 0.001, 0.001, 0.001, 0...."
-0x67e79051c947ed4d,"[[0.35335335, 0.664, 0.06, 0.028, 0.057057057,..."


## Saving our preprocesssed data

Let's save our anndata so we don't have to run these analyses again.  
We can't use standard Anndata functionality for this, since we're storing numpy arrays in both our .obs and .var. This is a logical way to structure our data, but is not allowed by Anndata when saving and loading.  
Therefore, we have our own I/O functions that are simple wrappers around Anndata's functions (wherein the numpy arrays are moved to and back from .uns, which does allow for arrays).  
You can save and load anndatas with {func}`~tfmindi.save_h5ad` and {func}`~tfmindi.load_h5ad`.
Be aware that these files can become large.

In [18]:
tm.save_h5ad(adata, "seqlets.h5ad")